In [ ]:
import sys
sys.path.append('../../../code/libs/')

%load_ext autoreload
%autoreload 2
import utils
import viz
import ios
import constants
import text as txtlib

In [2]:
# import os
import pandas as pd
import numpy as np

In [3]:
ROOT = 'births_and_deaths_2015_2022'
QUERY_Y = "year>=2015 and year<=2022"

In [4]:
# state_name	po_total	po_male	
ios.get_files_from_pattern(ios.path_join(ROOT,"*.txt"))

['births_and_deaths_2015_2022/Underlying Cause of Death, 2018-2022.txt',
 'births_and_deaths_2015_2022/Underlying Cause of Death, 1999-2020.txt',
 'births_and_deaths_2015_2022/Fetal Deaths, 2014-2022 expanded.txt',
 'births_and_deaths_2015_2022/Natality, 2007-2023.txt',
 'births_and_deaths_2015_2022/Underlying Cause of Death Drugs, 2018-2022.txt',
 'births_and_deaths_2015_2022/Underlying Cause of Death Drugs, 1999-2020.txt']

# Fetal Deaths & Births

In [5]:
files_f = ios.get_files_from_pattern(ios.path_join(ROOT,"Fetal*.txt"))
files_n = ios.get_files_from_pattern(ios.path_join(ROOT,"Natality*.txt"))
files = files_f + files_n

data1 = None

for fn in files:
    tmp = ios.read_csv(fn, index_col=[1,3], sep='\t').drop(columns=['Notes','Standard Residence States Code','Year Code', "State Code"], errors='ignore')
    tmp.rename(columns={c:c.lower().replace(' ','_').replace('%','percent') for c in tmp.columns}, inplace=True)
    tmp.index.names = ['state_name', 'year']
    tmp = tmp[tmp.index.get_level_values('year').notna()]
    tmp.index = tmp.index.set_levels(tmp.index.levels[1].map(int),level='year')

    data1 = tmp.copy() if data1 is None else data1.join(tmp)

data1 = data1.rename(columns={"fetal_deaths":'deaths_fetal',
                      "percent_of_total_deaths":'deaths_percent',
                      "births":"births_total",
                      'birth_rate':'births_rate',
                      "percent_of_total_births":"births_percent",
                      "average_birth_weight":"births_avg_weight",
                      "average_age_of_mother":"births_avg_age_mother",
                     }).drop(columns=['total_population','female_population'])
data1.shape

(459, 8)

In [6]:
data1 = data1.reset_index().query(QUERY_Y).set_index(['state_name','year']) #51
data1

deaths_fetal  deaths_percent  births_total births_percent  \
state_name year                                                              
Alabama    2015         544.0            0.27       59657.0          0.09%   
           2016         558.0            0.28       59151.0          0.09%   
           2017         545.0            0.27       58941.0          0.09%   
           2018         497.0            0.25       57761.0          0.09%   
           2019         526.0            0.26       58615.0          0.09%   
...                       ...             ...           ...            ...   
Wyoming    2018          40.0            0.02        6562.0          0.01%   
           2019          40.0            0.02        6565.0          0.01%   
           2020          31.0            0.02        6128.0          0.01%   
           2021          53.0            0.03        6237.0          0.01%   
           2022          36.0            0.02        6049.0          0.01%   

                   births_rate fertility_rate  births_avg_weight  \
state_name year                                                    
Alabama    2015          12.28          62.21            3197.72   
           2016          12.16          62.08            3192.66   
           2017          12.09          62.05            3189.11   
           2018          11.82          60.88            3184.39   
           2019          11.95          61.74            3181.87   
...                        ...            ...                ...   
Wyoming    2018          11.36          61.04            3173.17   
           2019          11.34          60.82            3160.35   
           2020          10.52          56.43            3167.60   
           2021  Not Available  Not Available            3173.62   
           2022  Not Available  Not Available            3165.51   

                 births_avg_age_mother  
state_name year                         
Alabama    2015                  27.03  
           2016                  27.18  
           2017                  27.29  
           2018                  27.42  
           2019                  27.46  
...                                ...  
Wyoming    2018                  28.06  
           2019                  28.11  
           2020                  28.10  
           2021                  28.29  
           2022                  28.37  

[408 rows x 8 columns]

# Death by Drugs

In [7]:
files = ios.get_files_from_pattern(ios.path_join(ROOT,"Underlying*Drugs,*.txt"))
data = pd.DataFrame()
data2 = []

for fn in files:
    tmp = ios.read_csv(fn, sep='\t').drop(columns=['Notes','Year Code', "State Code"], errors='ignore').reset_index(drop=True)
    tmp.rename(columns={c:c.lower().replace(' ','_') for c in tmp.columns}, inplace=True)
    tmp.rename(columns={'state':'state_name', 'deaths':'drug_deaths'}, inplace=True)
    
    tmp = tmp.groupby(['state_name','year']).sum()
    
    tmp = tmp[tmp.index.get_level_values('year').notna()]
    tmp.index = tmp.index.set_levels(tmp.index.levels[1].map(int),level='year')
    tmp = tmp['drug_deaths']
    
    print(tmp.shape)
    data2.append(tmp)

data2 = pd.concat(data2, sort=True, ignore_index=False)
data2 = data2.reset_index().sort_values(['state_name','year']).query(QUERY_Y)
data2 = data2.drop_duplicates(subset=['state_name','year'])
data2 = data2.set_index(['state_name','year'])
data2.shape

(255,)
(306,)


(408, 1)

In [8]:
data2

drug_deaths
state_name year             
Alabama    2015       1579.0
           2016       1659.0
           2017       1871.0
           2018       1760.0
           2019       1754.0
...                      ...
Wyoming    2018        121.0
           2019        149.0
           2020        187.0
           2021        219.0
           2022        236.0

[408 rows x 1 columns]

# Cause of Death

In [9]:
# files = ios.get_files_from_pattern(ios.path_join(ROOT,"Underlying*.txt"))
# data = pd.DataFrame()
# data2 = []

# for fn in files:
#     tmp = ios.read_csv(fn, sep='\t').drop(columns=['Notes','Year Code', "State Code"], errors='ignore').reset_index(drop=True)
#     tmp.rename(columns={c:c.lower().replace(' ','_') for c in tmp.columns}, inplace=True)
#     tmp.rename(columns={'state':'state_name'}, inplace=True)
    
#     death_cause_codes = ['F19.1', 'F19.9'] # drug abuse
#     tmp = tmp.query("cause_of_death_code in @death_cause_codes").groupby(['state_name','year']).sum()
    
#     tmp = tmp[tmp.index.get_level_values('year').notna()]
#     tmp.index = tmp.index.set_levels(tmp.index.levels[1].map(int),level='year')
#     tmp.drop(columns=['population'], inplace=True)
    
#     print(tmp.shape)
#     data2.append(tmp)

# data2 = pd.concat(data2, sort=True, ignore_index=False)
# data2 = data2.reset_index().sort_values(['state_name','year']).query(QUERY_Y)
# data2 = data2.drop_duplicates(subset=['state_name','year'])
# data2 = data2.set_index(['state_name','year'])
# data2.shape

# Final

In [10]:
data = data1.join(data2, how='left')
data.shape

(408, 9)

In [11]:
data.loc[:,'births_percent'] = data.loc[:,'births_percent'].apply(lambda v: float(str(v).replace('%','')))
data.loc[:,'births_rate'] = data.loc[:,'births_rate'].apply(lambda v: None if v=='Not Available' else v)
data.loc[:,'fertility_rate'] = data.loc[:,'fertility_rate'].apply(lambda v: None if v=='Not Available' else v)

In [12]:
data

deaths_fetal  deaths_percent  births_total  births_percent  \
state_name year                                                               
Alabama    2015         544.0            0.27       59657.0            0.09   
           2016         558.0            0.28       59151.0            0.09   
           2017         545.0            0.27       58941.0            0.09   
           2018         497.0            0.25       57761.0            0.09   
           2019         526.0            0.26       58615.0            0.09   
...                       ...             ...           ...             ...   
Wyoming    2018          40.0            0.02        6562.0            0.01   
           2019          40.0            0.02        6565.0            0.01   
           2020          31.0            0.02        6128.0            0.01   
           2021          53.0            0.03        6237.0            0.01   
           2022          36.0            0.02        6049.0            0.01   

                births_rate fertility_rate  births_avg_weight  \
state_name year                                                 
Alabama    2015       12.28          62.21            3197.72   
           2016       12.16          62.08            3192.66   
           2017       12.09          62.05            3189.11   
           2018       11.82          60.88            3184.39   
           2019       11.95          61.74            3181.87   
...                     ...            ...                ...   
Wyoming    2018       11.36          61.04            3173.17   
           2019       11.34          60.82            3160.35   
           2020       10.52          56.43            3167.60   
           2021        None           None            3173.62   
           2022        None           None            3165.51   

                 births_avg_age_mother  drug_deaths  
state_name year                                      
Alabama    2015                  27.03       1579.0  
           2016                  27.18       1659.0  
           2017                  27.29       1871.0  
           2018                  27.42       1760.0  
           2019                  27.46       1754.0  
...                                ...          ...  
Wyoming    2018                  28.06        121.0  
           2019                  28.11        149.0  
           2020                  28.10        187.0  
           2021                  28.29        219.0  
           2022                  28.37        236.0  

[408 rows x 9 columns]

In [13]:
ios.save_csv(data, 'births_and_deaths_2015_2022.csv')